In [1]:
!wget https://homl.info/shakespeare

--2025-11-05 20:58:29--  https://homl.info/shakespeare
Resolving homl.info (homl.info)... 155.133.138.14, 2001:4b98:dc5:253::14
Connecting to homl.info (homl.info)|155.133.138.14|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/karpathy/char-rnn/raw/master/data/tinyshakespeare/input.txt [following]
--2025-11-05 20:58:30--  https://github.com/karpathy/char-rnn/raw/master/data/tinyshakespeare/input.txt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt [following]
--2025-11-05 20:58:30--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.github

In [2]:
with open('/content/shakespeare', 'r') as f:
    shakespeare_text = f.read()

In [3]:
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import numpy as np
from tqdm import tqdm, trange
from torch.utils.data import TensorDataset, DataLoader

In [4]:
unique_words = sorted(set(shakespeare_text.split()))
words = {word: idx for idx, word in enumerate(unique_words)}
shakespeare_tokens = [words[word] for word in shakespeare_text.split()]

In [5]:
print(shakespeare_text.split()[:10])
print(shakespeare_tokens[:10])

['First', 'Citizen:', 'Before', 'we', 'proceed', 'any', 'further,', 'hear', 'me', 'speak.']
[1580, 992, 637, 24643, 18488, 5159, 11877, 12900, 15681, 21344]


In [6]:
X_sequences = []
y_sequences = []
sequence_length = 100
shift = 1

for i in range(0, len(shakespeare_tokens) - sequence_length, shift):
    x = shakespeare_tokens[i:i+sequence_length]
    y = shakespeare_tokens[i+sequence_length]
    X_sequences.append(x)
    y_sequences.append(y)

In [7]:
class Model(nn.Module):
    def __init__(self, vocab_size, embed_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru1 = nn.GRU(embed_dim, 128, batch_first=True)
        self.gru2 = nn.GRU(128, 256, batch_first=True)
        self.output = nn.Linear(256, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.gru1(x)
        x = F.dropout(x, 0.2)
        x, _ = self.gru2(x)
        x = F.dropout(x, 0.2)
        x = x[:, -1]
        return self.output(x)

In [8]:
def train_model(model, optimizer, criterion, train_loader, epochs, device):
    model.to(device)
    model.train()
    losses = []

    for epoch in trange(epochs, desc="Epochs"):
        total_loss = 0

        for inp, target in train_loader:
            inp = inp.to(device)
            target = target.to(device)

            optimizer.zero_grad()
            logits = model(inp)  # (B, V)

            loss = criterion(logits, target)  # target (B,)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} - loss: {avg_loss:.4f}")

    return losses

In [9]:
X_tensor = torch.tensor(X_sequences, dtype=torch.long)
y_tensor = torch.tensor(y_sequences, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)

BATCH_SIZE = 32
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
vocab_size = len(words)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = Model(vocab_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

epochs = 20

history = train_model(model,
                      optimizer,
                      criterion,
                      train_loader,
                      epochs,
                      device)

In [ ]:
def tokenizer(text):
    token_ids = [words[word] for word in text.split()]
    return torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(device)

In [ ]:
@torch.no_grad()
def next_token(input_ids, temperature=1.0):
    logits = model(input_ids)          # (1, vocab_size)
    logits = logits / temperature
    probs = torch.softmax(logits, dim=-1)
    next_id = torch.multinomial(probs, 1)  # (1,1)
    return next_id

In [ ]:
# input_seq = torch.tensor(tokenizer("I love you"), dtype=torch.long).unsqueeze(0).to(device)
input_seq = tokenizer("I love you")
next_token(input_seq)

In [ ]:
def decode(input_ids):
    idx2word = {idx: word for word, idx in words.items()}
    words_list = [idx2word[i] for i in input_ids]
    return " ".join(words_list)

In [ ]:
def generate(text, max_new_tokens=50, temperature=1.0):
    input_ids =  tokenizer(text)
    for _ in range(max_new_tokens):
        next_id = next_token(input_ids, temperature)
        input_ids = torch.cat([input_ids, next_id], dim=1)

    return decode(input_ids[0].tolist())

In [ ]:
generated_text = generate("First Citizen: Before we proceed", max_new_tokens=50, temperature=1.0)
print(generated_text)

In [ ]:
class StatefulGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128, num_layers=2):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, embed_dim)

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.gru = nn.GRU(embed_dim, hidden_size, num_layers=num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)

        self.hidden = None

    def reset_state(self, batch_size=1, device="cuda"):
        self.hidden = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(device)

    def forward(self, x):

        if self.hidden is None:
            self.reset_state(batch_size=x.size(0), device=x.device)

        self.hidden = self.hidden.detach()

        x = self.embed(x)
        out, self.hidden = self.gru(x, self.hidden)
        logits = self.linear(out[:, -1])

        return logits

In [ ]:
class ResetStateCallback:
    def __init__(self, model):
        self.model = model

    def on_epoch_begin(self):
        self.model.reset_state()

In [ ]:
def train_model(
    model,
    train_loader,
    optimizer,
    criterion,
    epochs,
    device="cuda",
    reset_each_batch=False
):

    callback = ResetStateCallback(model)

    model.to(device)
    model.train()

    for epoch in trange(epochs, desc=f"Train"):
        callback.on_epoch_begin()

        total_loss = 0

        for batch_idx, (x, y) in enumerate(train_loader):
            model.reset_state(batch_size=x.size(0), device=device)
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | loss: {total_loss/len(train_loader):.4f}")

In [ ]:
vocab_size = len(words)
model = StatefulGRU(vocab_size)
train_loader = DataLoader(dataset, batch_size=32, shuffle=False)
epochs = 20
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

train_model(
    model,
    train_loader,
    optimizer,
    criterion,
    epochs)

In [ ]:
def generate(text, max_new_tokens=50, temperature=1.0):
    # converte pra tensor e adiciona batch dimension
    input_ids = tokenizer(text) # shape (1, seq_len)

    model.reset_state(batch_size=1, device=device)  # garante hidden state correto

    for _ in range(max_new_tokens):
        next_id = next_token(input_ids, temperature)  # shape (1,1)
        input_ids = torch.cat([input_ids, next_id], dim=1)

    return decode(input_ids[0].tolist())


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
generated_text = generate("First Citizen: Before we proceed", max_new_tokens=50, temperature=1.0)
print(generated_text)